# 08 · Complex types: structs, arrays, maps & JSON

Real-world data (especially JSON from APIs and event logs) is nested. Spark has three complex types:

| Type | Like | Example |
|---|---|---|
| `struct` | a nested row / Python object with fixed fields | `device: {os: "iOS", version: "17.1"}` |
| `array` | a Python list (all elements same type) | `tags: ["sale", "new"]` |
| `map` | a Python dict (keys same type, values same type) | `properties: {"page": "/home"}` |

In [1]:
# Setup — run this cell first every time you open the notebook
import os, sys
os.environ.setdefault("JAVA_HOME", "/opt/homebrew/opt/openjdk@17")   # Spark runs on Java
os.environ["PYSPARK_PYTHON"] = sys.executable                         # workers use this same Python

from pyspark.sql import SparkSession, Window, functions as F, types as T

spark = (SparkSession.builder
         .appName("pyspark-course")
         .master("local[*]")                               # run locally, use all CPU cores
         .config("spark.sql.shuffle.partitions", "8")      # explained in lesson 10
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("FATAL")                   # keep Java's log noise out of the notebook

DATA, OUT = "../data", "../output"

In [2]:
events = spark.read.json(f"{DATA}/events.jsonl")
events.printSchema()
events.show(3, truncate=False)

root
 |-- device: struct (nullable = true)
 |    |-- os: string (nullable = true)
 |    |-- version: string (nullable = true)
 |-- event: string (nullable = true)
 |-- properties: struct (nullable = true)
 |    |-- page: string (nullable = true)
 |    |-- referrer: string (nullable = true)
 |-- tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- ts: string (nullable = true)
 |-- user_id: long (nullable = true)

+---------------+-----------+----------------------+-----------------------+-------------------+-------+
|device         |event      |properties            |tags                   |ts                 |user_id|
+---------------+-----------+----------------------+-----------------------+-------------------+-------+
|{Android, 13.5}|click      |{/product, direct}    |[email, promo]         |2024-06-22 17:09:14|107    |
|{iOS, 14.6}    |view       |{/product, newsletter}|[organic, mobile, sale]|2024-06-14 01:44:47|36     |
|{macOS, 10.4}  |add_to_cart|

## Structs
Reach inside with a dot. `"device.*"` expands all fields into top-level columns.

In [3]:
events.select("user_id", "device.os", F.col("device.version").alias("os_version")).show(3)
events.select("user_id", "device.*").show(3)

+-------+-------+----------+
|user_id|     os|os_version|
+-------+-------+----------+
|    107|Android|      13.5|
|     36|    iOS|      14.6|
|     75|  macOS|      10.4|
+-------+-------+----------+
only showing top 3 rows
+-------+-------+-------+
|user_id|     os|version|
+-------+-------+-------+
|    107|Android|   13.5|
|     36|    iOS|   14.6|
|     75|  macOS|   10.4|
+-------+-------+-------+
only showing top 3 rows


## Arrays

In [4]:
events.select(
    "tags",
    F.size("tags").alias("n_tags"),
    F.array_contains("tags", "sale").alias("has_sale"),
    F.col("tags")[0].alias("first_tag"),                 # 0-based; null if missing
    F.sort_array("tags").alias("sorted"),
).show(5, truncate=False)

+-----------------------+------+--------+---------+-----------------------+
|tags                   |n_tags|has_sale|first_tag|sorted                 |
+-----------------------+------+--------+---------+-----------------------+
|[email, promo]         |2     |false   |email    |[email, promo]         |
|[organic, mobile, sale]|3     |true    |organic  |[mobile, organic, sale]|
|[promo, email, organic]|3     |false   |promo    |[email, organic, promo]|
|[organic]              |1     |false   |organic  |[organic]              |
|[organic]              |1     |false   |organic  |[organic]              |
+-----------------------+------+--------+---------+-----------------------+
only showing top 5 rows


### `explode`: one row per array element
This is how you "flatten" arrays to analyse their contents. `explode` **drops** rows whose array is empty
or null; `explode_outer` keeps them (with a null). `posexplode` also gives the position.

In [5]:
small = events.select("user_id", "tags").limit(4)
small.show(truncate=False)
small.select("user_id", F.explode("tags").alias("tag")).show()
small.select("user_id", F.explode_outer("tags").alias("tag")).show()

+-------+-----------------------+
|user_id|tags                   |
+-------+-----------------------+
|107    |[email, promo]         |
|36     |[organic, mobile, sale]|
|75     |[promo, email, organic]|
|134    |[organic]              |
+-------+-----------------------+



+-------+-------+
|user_id|    tag|
+-------+-------+
|    107|  email|
|    107|  promo|
|     36|organic|
|     36| mobile|
|     36|   sale|
|     75|  promo|
|     75|  email|
|     75|organic|
|    134|organic|
+-------+-------+

+-------+-------+
|user_id|    tag|
+-------+-------+
|    107|  email|
|    107|  promo|
|     36|organic|
|     36| mobile|
|     36|   sale|
|     75|  promo|
|     75|  email|
|     75|organic|
|    134|organic|
+-------+-------+



## Maps
Spark's JSON reader inferred `properties` as a *struct*. If the keys vary from record to record, a **map**
is the better type — and we can request it with an explicit schema:

In [6]:
schema = """
    user_id int, ts timestamp, event string,
    device struct<os: string, version: string>,
    tags array<string>,
    properties map<string, string>
"""
events_m = spark.read.schema(schema).json(f"{DATA}/events.jsonl")
events_m.select(
    "properties",
    F.col("properties")["page"].alias("page"),
    F.map_keys("properties").alias("keys"),
).show(3, truncate=False)

events_m.select("user_id", F.explode("properties").alias("key", "value")).show(4)   # map → key/value rows

+------------------------------------------+--------+----------------+
|properties                                |page    |keys            |
+------------------------------------------+--------+----------------+
|{page -> /product, referrer -> direct}    |/product|[page, referrer]|
|{page -> /product, referrer -> newsletter}|/product|[page, referrer]|
|{page -> /product, referrer -> direct}    |/product|[page, referrer]|
+------------------------------------------+--------+----------------+
only showing top 3 rows
+-------+--------+----------+
|user_id|     key|     value|
+-------+--------+----------+
|    107|    page|  /product|
|    107|referrer|    direct|
|     36|    page|  /product|
|     36|referrer|newsletter|
+-------+--------+----------+
only showing top 4 rows


## Building complex columns

In [7]:
events.select(
    F.struct("event", "user_id").alias("as_struct"),
    F.array(F.lit(1), F.lit(2), F.lit(3)).alias("as_array"),
    F.create_map(F.lit("os"), F.col("device.os")).alias("as_map"),
).show(2, truncate=False)

# and the reverse of explode: collect back into arrays
events.groupBy("user_id").agg(F.collect_set("event").alias("events_seen")).show(3, truncate=False)

+------------+---------+---------------+
|as_struct   |as_array |as_map         |
+------------+---------+---------------+
|{click, 107}|[1, 2, 3]|{os -> Android}|
|{view, 36}  |[1, 2, 3]|{os -> iOS}    |
+------------+---------+---------------+
only showing top 2 rows


+-------+------------------------------------+
|user_id|events_seen                         |
+-------+------------------------------------+
|1      |[view, click, add_to_cart]          |
|2      |[view, purchase, click, add_to_cart]|
|3      |[view, purchase, click, add_to_cart]|
+-------+------------------------------------+
only showing top 3 rows


## JSON strings inside columns
Sometimes a column holds raw JSON **text** (common with Kafka messages or database dumps).
* `from_json(col, schema)` — parse it into a struct
* `to_json(col)` — serialise a struct/array/map back to a string
* `get_json_object(col, "$.path")` — pull out one field without a schema

In [8]:
raw = spark.createDataFrame([('{"name": "Ann", "langs": ["py", "sql"]}',),
                             ('{"name": "Ben", "langs": ["scala"]}',)], "payload string")
parsed = raw.select(F.from_json("payload", "name string, langs array<string>").alias("p"))
parsed.select("p.name", "p.langs").show()
raw.select(F.get_json_object("payload", "$.name").alias("name")).show()
events.select(F.to_json("device").alias("device_json")).show(2, truncate=False)

+----+---------+
|name|    langs|
+----+---------+
| Ann|[py, sql]|
| Ben|  [scala]|
+----+---------+

+----+
|name|
+----+
| Ann|
| Ben|
+----+



+---------------------------------+
|device_json                      |
+---------------------------------+
|{"os":"Android","version":"13.5"}|
|{"os":"iOS","version":"14.6"}    |
+---------------------------------+
only showing top 2 rows


## Higher-order functions: transform arrays without exploding
Pass a Python `lambda` that receives Spark Columns:

In [9]:
nums = spark.createDataFrame([([1, 2, 3, 4],), ([10, 20],)], "xs array<int>")
nums.select(
    "xs",
    F.transform("xs", lambda x: x * 10).alias("times10"),
    F.filter("xs", lambda x: x % 2 == 0).alias("evens"),
    F.exists("xs", lambda x: x > 15).alias("any_gt_15"),
    F.aggregate("xs", F.lit(0), lambda acc, x: acc + x).alias("total"),
).show()

+------------+----------------+--------+---------+-----+
|          xs|         times10|   evens|any_gt_15|total|
+------------+----------------+--------+---------+-----+
|[1, 2, 3, 4]|[10, 20, 30, 40]|  [2, 4]|    false|   10|
|    [10, 20]|      [100, 200]|[10, 20]|     true|   30|
+------------+----------------+--------+---------+-----+



### Exercise 1
Create `events_per_os`: columns **`os`, `n`** — the number of events per device OS.

In [10]:
events_per_os = events.groupBy(F.col("device.os").alias("os")).agg(F.count("*").alias("n"))

In [11]:
# Check — run me after your answer
assert sorted(events_per_os.columns) == ["n", "os"]
assert {r.os: r.n for r in events_per_os.collect()} == {'iOS': 508, 'Android': 505, 'Windows': 502, 'macOS': 485}
print("Correct!")

Correct!


### Exercise 2
Create `tag_counts`: columns **`tag`, `n`** — how many times each tag appears across all events.

In [12]:
tag_counts = events.select(F.explode("tags").alias("tag")).groupBy("tag").agg(F.count("*").alias("n"))

In [13]:
# Check — run me after your answer
assert {r.tag: r.n for r in tag_counts.collect()} == {'promo': 471, 'organic': 459, 'email': 451, 'new': 448, 'mobile': 444, 'sale': 420}
print("Correct!")

Correct!


### Exercise 3
Store in `no_tags` the number of events whose `tags` array is **empty**.

In [14]:
no_tags = events.filter(F.size("tags") == 0).count()

In [15]:
# Check — run me after your answer
assert no_tags == 391, no_tags
print("Correct!")

Correct!


### Exercise 4
Parse `orders_json` (below is already in the cell) into `parsed_orders` with **flat** columns **`sku`, `qty`, `w`, `h`** (`qty`, `w`, `h` as ints).

In [16]:
orders_json = spark.createDataFrame([
    ('{"sku": "A1", "qty": 2, "dims": {"w": 10, "h": 5}}',),
    ('{"sku": "B7", "qty": 1, "dims": {"w": 3, "h": 8}}',),
], "payload string")

schema = "sku string, qty int, dims struct<w: int, h: int>"
parsed_orders = (orders_json.select(F.from_json("payload", schema).alias("o"))
                            .select("o.sku", "o.qty", "o.dims.w", "o.dims.h"))

In [17]:
# Check — run me after your answer
assert parsed_orders.columns == ["sku", "qty", "w", "h"]
assert [tuple(r) for r in parsed_orders.collect()] == [("A1", 2, 10, 5), ("B7", 1, 3, 8)]
print("Correct!")

Correct!


### Exercise 5
Create `upper_tags`: `events` plus a column **`tags_upper`** with every tag upper-cased — **without** exploding (use `F.transform`).

In [18]:
upper_tags = events.withColumn("tags_upper", F.transform("tags", lambda t: F.upper(t)))

In [19]:
# Check — run me after your answer
r = upper_tags.filter(F.size("tags") > 1).first()
assert r.tags_upper == [t.upper() for t in r.tags]
print("Correct!")

Correct!
